This notebook converts Label Studio annotations into Ultralytics YOLO pose format, extracts the corresponding video-frame images, and creates a grouped training/validation split. It then fine-tunes a YOLOv8 model to detect and localize predator and prey fish in frames.

### Fish Detector Training

In [ ]:
# import libraries + set working directory
import os, sys, shutil, json, random
from urllib.parse import urlparse, parse_qs

DRIVE_ROOT = "/path/to/project"
source_dir = os.path.join(DRIVE_ROOT, "Data/1. Data Processing/Raw/Fish Detector YOLO")

WORK_DIR = os.path.expanduser("~/fish_detector_work")
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

os.makedirs('dataset', exist_ok=True)
shutil.copy(os.path.join(source_dir, 'training_export.json'), 'training_export.json')
shutil.copy(os.path.join(source_dir, 'images.tar'), 'dataset/images.tar')

'dataset/images.tar'

In [ ]:
# create data.yaml
with open("dataset/data.yaml", "w") as f:
    f.write("""path: dataset
train: train.txt
val: val.txt
nc: 2
names: [Predator, Prey]
kpt_shape: [1, 3]
""")

# extract images into the right place
os.makedirs("dataset/images", exist_ok=True)
!tar -xvf dataset/images.tar -C dataset/images

pred_attack_1/
pred_attack_1/frame_006.jpg
pred_attack_1/frame_086.jpg
pred_attack_1/frame_040.jpg
pred_attack_1/frame_076.jpg
pred_attack_1/frame_014.jpg
pred_attack_1/frame_094.jpg
pred_attack_1/frame_036.jpg
pred_attack_1/frame_044.jpg
pred_attack_1/frame_110.jpg
pred_attack_1/frame_038.jpg
pred_attack_1/frame_116.jpg
pred_attack_1/frame_046.jpg
pred_attack_1/frame_018.jpg
pred_attack_1/frame_082.jpg
pred_attack_1/frame_052.jpg
pred_attack_1/frame_026.jpg
pred_attack_1/frame_008.jpg
pred_attack_1/frame_100.jpg
pred_attack_1/frame_072.jpg
pred_attack_1/frame_022.jpg
pred_attack_1/frame_056.jpg
pred_attack_1/frame_112.jpg
pred_attack_1/frame_054.jpg
pred_attack_1/frame_064.jpg
pred_attack_1/frame_042.jpg
pred_attack_1/frame_000.jpg
pred_attack_1/frame_048.jpg
pred_attack_1/frame_058.jpg
pred_attack_1/frame_002.jpg
pred_attack_1/frame_034.jpg
pred_attack_1/frame_090.jpg
pred_attack_1/frame_104.jpg
pred_attack_1/frame_124.jpg
pred_attack_1/frame_084.jpg
pred_attack_1/frame_068.jpg
pred_

In [ ]:
# convert Label Studio export to YOLO label files
with open("training_export.json") as f:
    data = json.load(f)

label_map = {"Predator": 0, "Prey": 1}

os.makedirs("dataset/labels", exist_ok=True)

for task in data:
    img_field = task["data"]["img"]

    parsed = urlparse(img_field)

    query = parse_qs(parsed.query)
    rel_path = query["d"][0].removeprefix("images/")
    txt_rel_path = os.path.splitext(rel_path)[0] + ".txt"

    annotations = task.get("annotations", [])
    if not annotations:
        continue

    results = annotations[0]["result"]
    lines = []

    for r in results:
        val = r["value"]
        label = val["keypointlabels"][0]
        class_id = label_map[label]
        x = val["x"] / 100
        y = val["y"] / 100
        box_size = 0.03

        lines.append(f"{class_id} {x:.6f} {y:.6f} {box_size:.6f} {box_size:.6f} {x:.6f} {y:.6f} 2")

    label_full_path = os.path.join("dataset/labels", txt_rel_path)
    os.makedirs(os.path.dirname(label_full_path), exist_ok=True)

    with open(label_full_path, "w") as out:
        out.write("\n".join(lines))

print("Conversion done.")

Conversion done.


In [ ]:
# build attack-level train/val split
images_root = os.path.abspath("dataset/images")
labels_root = os.path.abspath("dataset/labels")

labeled_attacks = set()
attack_to_images = {}

for root, _, files in os.walk(images_root):
    attack_name = os.path.relpath(root, images_root).split(os.sep)[0]
    for f in files:
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        rel = os.path.relpath(os.path.join(root, f), images_root)
        label_path = os.path.join(labels_root, os.path.splitext(rel)[0] + ".txt")
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            labeled_attacks.add(attack_name)
            attack_to_images.setdefault(attack_name, []).append(os.path.join(images_root, rel))

labeled_attacks = sorted(labeled_attacks)
print(f"Attacks with labels: {labeled_attacks}")
print(f"Counts per attack: {[(a, len(attack_to_images[a])) for a in labeled_attacks]}")

random.seed(42)
random.shuffle(labeled_attacks)

n_val_attacks = max(1, int(len(labeled_attacks) * 0.2))
val_attacks = set(labeled_attacks[:n_val_attacks])

train_imgs, val_imgs = [], []

for attack_name in labeled_attacks:
    target = val_imgs if attack_name in val_attacks else train_imgs
    target.extend(attack_to_images[attack_name])

with open("dataset/train.txt", "w") as f:
    f.write("\n".join(train_imgs))
with open("dataset/val.txt", "w") as f:
    f.write("\n".join(val_imgs))

print(f"Val attacks: {val_attacks}")
print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}")

assert len(train_imgs) > 0, "Train set is empty!"
assert len(val_imgs) > 0, "Val set is empty!"

Attacks with labels: ['pred_attack_1', 'pred_attack_10', 'pred_attack_11', 'pred_attack_12', 'pred_attack_13', 'pred_attack_14', 'pred_attack_15', 'pred_attack_16', 'pred_attack_17', 'pred_attack_18', 'pred_attack_19', 'pred_attack_2', 'pred_attack_20', 'pred_attack_21', 'pred_attack_22', 'pred_attack_23', 'pred_attack_24', 'pred_attack_25', 'pred_attack_26', 'pred_attack_27', 'pred_attack_3', 'pred_attack_4', 'pred_attack_5', 'pred_attack_6', 'pred_attack_7', 'pred_attack_8', 'pred_attack_9']
Counts per attack: [('pred_attack_1', 63), ('pred_attack_10', 63), ('pred_attack_11', 63), ('pred_attack_12', 63), ('pred_attack_13', 63), ('pred_attack_14', 63), ('pred_attack_15', 63), ('pred_attack_16', 63), ('pred_attack_17', 63), ('pred_attack_18', 63), ('pred_attack_19', 63), ('pred_attack_2', 63), ('pred_attack_20', 63), ('pred_attack_21', 63), ('pred_attack_22', 63), ('pred_attack_23', 63), ('pred_attack_24', 63), ('pred_attack_25', 63), ('pred_attack_26', 63), ('pred_attack_27', 63), ('p

In [ ]:
# train (once, with early stopping)
!pip install ultralytics

from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")
model.train(data="dataset/data.yaml", epochs=100, imgsz=640, batch=16, patience=15)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.115 🚀 Python-3.10.20 torch-2.5.1+rocm6.2 CUDA:0 (AMD Radeon RX 6700 XT, 12272MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, form

/home/user/.local/lib/python3.10/site-packages/ultralytics/nn/modules/block.py:1325: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  attn = (q * self.scale).transpose(-2, -1) @ k
/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_ones(w, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_determinis

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2401.9±821.4 MB/s, size: 49.9 KB)
train: Scanning /path/to/workdir/dataset/labels/pred_attack_1.cache... 1386 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1386/1386 342.0Mit/s 0.0s
WARNING ⚠️ No 'flip_idx' array defined in data.yaml, disabling 'fliplr' and 'flipud' augmentations.
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1484.4±674.1 MB/s, size: 50.5 KB)
val: Scanning /path/to/workdir/dataset/labels/pred_attack_18.cache... 315 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 315/315 15.9Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 63 weight(decay=0.0), 73 weight(decay=0.0005), 72 bias(decay=0.0)
Plotting labels to /path/to/workdir/runs/pose/train-4/labels.jpg... 
Image sizes 640 train, 640 val
Using 6 dataloader worker

/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_ones(w, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feats[0].new_ones(h, dtype=dtype).cumsum(0) - (1 - grid_ce


      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
      1/100     0.207G      2.825     0.2694     0.3393      2.285      1.376        227        640: 100% ━━━━━━━━━━━━ 87/87 4.5it/s 19.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.3it/s 1.9s.2s
                   all        315       5355      0.507      0.537      0.489     0.0816      0.569      0.602      0.591      0.588

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
      2/100      1.99G      1.937    0.08324    0.08987      1.207     0.9587        399        640: 0% ──────────── 0/87  0.2s

...

/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_ones(w, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feats[0].new_ones(h, dtype=dtype).cumsum(0) - (1 - grid_ce

     51/100      2.52G     0.7387   0.009165   0.009455     0.3935     0.8011        174        640: 100% ━━━━━━━━━━━━ 87/87 5.8it/s 15.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.6it/s 1.5s.2s
                   all        315       5355      0.989      0.964      0.992      0.776      0.991      0.966      0.993      0.993

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
     52/100      2.52G     0.6874    0.00665    0.00846     0.3682     0.8029        285        640: 0% ──────────── 0/87  0.2s

/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_ones(w, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/user/.local/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feats[0].new_ones(h, dtype=dtype).cumsum(0) - (1 - grid_ce

     52/100      2.52G     0.7298   0.009017    0.01016     0.3918     0.8027        196        640: 100% ━━━━━━━━━━━━ 87/87 5.8it/s 15.0s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.6it/s 1.5s.2s
                   all        315       5355      0.988      0.962      0.991      0.762       0.99      0.963      0.992      0.992
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 37, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

52 epochs completed in 0.248 hours.
Optimizer stripped from /path/to/workdir/runs/pose/train-4/weights/last.pt, 6.4MB
Optimizer stripped from /path/to/workdir/runs/pose/train-4/weights/best.pt, 6.4MB

Validating /path/to/workdir/runs/pose/train-4/weights/best.pt...
Ultra

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8644671f00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [ ]:
# back up finished results to drive
RUN_DIR = os.path.join(WORK_DIR, "runs", "pose", "train-4")
DEST = os.path.join(source_dir, "runs_pose_train_final")
shutil.copytree(RUN_DIR, DEST, dirs_exist_ok=True)
print(f"Backed up training run to {DEST}")

Backed up training run to /path/to/project/Data/1. Data Processing/Raw/Fish Detector YOLO/runs_pose_train_final
